# Spindle Orientation Simulator

This notebook lets you run the spindle-orientation simulation model interactively — pick a cell type and a few parameters with sliders and dropdowns, click **Run simulation**, and see the result. No coding or knowledge of the underlying parameter-encoding scheme required.

**What the model simulates:** during cell division, the mitotic spindle often needs to rotate to a specific angle before the cell divides. This model treats that as a physical tug-of-war at the cell cortex (the cell's outer boundary):

- **Astral microtubules** grow out from each spindle pole toward the cortex. When one reaches the cortex, it can **push** the spindle pole away.
- **Cortical force generators ("motors")** sit at specific points on the cortex. When a microtubule tip is captured by one, the motor can **pull** that spindle pole toward it.

The balance of pushing and pulling, together with the shape of the cell and where the motors sit, determines how the spindle rotates and drifts over time.

Three previously-published cell/organism systems are supported:
- **Fly follicular epithelium** and **neuroblast** cells (idealized elliptical cell)
- **_C. elegans_ embryos** — pronuclei centering (PNC), or metaphase/anaphase spindle positioning (idealized cell)
- **Zebrafish embryonic cells** — using real cell shapes tracked from live-imaging movies (a handful of example tracked cells ship with this notebook)

Run the two setup cells below once per session, then use the widgets to configure and run a simulation.

In [ ]:
# --- Setup: works both locally and on Google Colab ---
import importlib.util, os, subprocess, sys

REQUIRED = ["cv2", "shapely", "openpyxl", "ipywidgets"]
missing = [pkg for pkg in REQUIRED if importlib.util.find_spec(pkg) is None]
if missing:
    print(f"Installing missing packages: {missing} ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "opencv-python-headless", "shapely", "openpyxl", "ipywidgets"], check=True)

# If the model file / data folders aren't next to this notebook (e.g. a fresh
# Colab session with the notebook opened on its own), clone the project repo.
# Fill this in once the project has been pushed to GitHub.
GITHUB_REPO_URL = ""  # e.g. "https://github.com/<user>/<repo>.git"

if not os.path.exists("spindle_model.py"):
    if GITHUB_REPO_URL:
        repo_dir = GITHUB_REPO_URL.rstrip("/").rsplit("/", 1)[-1]
        if repo_dir.endswith(".git"):
            repo_dir = repo_dir[:-4]
        if not os.path.exists(repo_dir):
            subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO_URL], check=True)
        os.chdir(repo_dir)
    else:
        raise FileNotFoundError(
            "spindle_model.py was not found next to this notebook, and "
            "GITHUB_REPO_URL is empty. Either run this notebook from inside the "
            "project folder, or set GITHUB_REPO_URL above to this project's "
            "GitHub URL once it has been published."
        )

print("Setup OK. Working directory:", os.getcwd())

In [ ]:
from spindle_model import run_simulation, build_ui, plot_summary, list_available_endo_cells

print("Tracked zebrafish cells available on this machine:", list_available_endo_cells())

### Quick glossary for the settings below

- **Number of motors / Motor density** — how many cortical force-generator "motors" are available to pull on the spindle. For the fly and _C. elegans_ cell types this is a plain count of motors placed on the cortex; for the zebrafish (tracked-cell) type it's a true density (motors per unit length of the real, tracked cortex perimeter), so the same value can correspond to a different absolute number of motors. The field label switches between "Number of motors" and "Motor density" depending on the cell type you pick.
- **Astral microtubules** — how many microtubules radiate from each spindle pole toward the cortex.
- **Spindle length** — for fly follicular epithelium/neuroblast, entered in µm (default 8 µm; the model rejects anything above 9.5 µm as too long for the cell). For _C. elegans_ spindle positioning, entered directly in the model's own length units (default 1.8).
- **Duration (s)** — how many seconds of simulated time to run. Pre-filled with the length used for the publication for the selected cell type (600s for fly cells, 360s/300s for _C. elegans_ PNC/spindle, or the tracked movie's own metaphase-to-anaphase duration for a zebrafish cell) — shorten it for a faster look, or lengthen it for a longer run.
- **Runtime estimate** — a rough, machine-dependent estimate of how long the *run itself* will take to compute, based on your duration and astral-MT settings (not to be confused with the simulated duration above). Recompute by adjusting the settings above.
- **Cortical pushing** — whether microtubules that reach the cortex are allowed to push the spindle pole away from it.
- **Cortical pulling (motors)** — whether motors that capture a microtubule tip are allowed to pull that spindle pole toward them.
- **Also save per-frame PDF plots + raw MT log to disk** — off by default (keeps runs fast and avoids cluttering disk with hundreds of PDFs per click). Turn on to reproduce the same frame-by-frame plots and raw microtubule log the original publication pipeline always writes.
- **Advanced (physical parameters)** — the underlying physical constants (microtubule catastrophe/rescue and growth/shrink rates, cortical friction, viscosity, spindle rigidity). Pre-filled with the model's published defaults; only change these if you know what you're doing.

In [3]:
from IPython.display import display

ui = build_ui()
display(ui)